# Productivización de modelos

Quizás uno de los aspectos clave es cómo poner en valor los modelos construidos para que tengan impacto en los procesos de negocio. Existen distintas modalidades en las que este proceso toma forma. Disponer de un entorno con garantías de qué modelo es el correcto a poner en marcha es quizás una de las claves a la hora de dar servicio a escala en la mayoría de las organizaciones. Veremos formas _manuales_ de hacerlo, pero es bueno que conozcamos las mejores prácticas en lo que respecta al servicio de modelos o _model serving_

En la actualidad muchas de estas plataformas se han especializado en dos modalidades, ML y Gen AI.


## MLFlow

Ampliaremos el ejercicio anteriormente realizado con Comet para el caso de MLFlow desplegado de forma local. MLFlow nos permite desplegar un servicio y actuar de forma local incluyendo el poder servir un modelo registrado en nuestro servidor de experimentos.

* https://mlflow.org/docs/latest/introduction/index.html

Una vez instalado podemos ejecutar nuestro servidor para que se quede "escuchando" en el puerto 5000. Deberemos abrir un terminal con el entorno python donde instalamos mlflow activo y ejecutar:

```sh
mlflow ui
```

No cerréis el terminal ya que el proceso se cerrará. Podéis acceder a la ruta http://127.0.0.1:5000/ para acceder a la interfaz local de vuestro sistema. Esto os permite configurar vuestro entorno Python para que emplee este registro como el punto en el que registrar nuestras métricas y modelos.

In [20]:
%pip install mlflow

In [21]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

Al igual que hicimos con Comet, podemos registrar las métricas que creamos relevantes para un experimento.

In [22]:
mlflow.set_experiment("check-localhost-connection")

with mlflow.start_run():
    mlflow.log_metric("foo", 1)
    mlflow.log_metric("bar", 2)

🏃 View run rebellious-fawn-385 at: http://localhost:5000/#/experiments/1/runs/7993b2c4019047b380273c1c84e6bd62
🧪 View experiment at: http://localhost:5000/#/experiments/1


Volver al interfaz para ver cómo un nuevo experimento fue registrado y las métricas asociadas a este. Veréis que no hay mucha magia ya que los datos como tal se registran en una carpeta en la ruta en la que estamos trabajando (revisad las carpetas _mlruns_ y _mlartifacts_).

In [23]:
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn

with mlflow.start_run() as run:
    X, y = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    params = {"max_depth": 2, "random_state": 42}
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    # Log parameters and metrics using the MLflow APIs
    mlflow.log_params(params)

    y_pred = model.predict(X_test)
    mlflow.log_metrics({"mse": mean_squared_error(y_test, y_pred)})

    # Log the sklearn model and register as version 1
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="sklearn-model",
        input_example=X_train,
        registered_model_name="sk-learn-random-forest-reg-model",
    )

2026/05/26 12:58:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/26 12:58:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'sk-learn-random-forest-reg-model' already exists. Creating a new version of this model...
2026/05/26 12:58:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: sk-learn-random-forest-reg-model, version 4


🏃 View run overjoyed-hound-678 at: http://localhost:5000/#/experiments/1/runs/249815cde7e1431899639f9bdc29b763
🧪 View experiment at: http://localhost:5000/#/experiments/1


Created version '4' of model 'sk-learn-random-forest-reg-model'.


Acabamos de registrar nuestro primer modelo http://127.0.0.1:5000/#/models/sk-learn-random-forest-reg-model. Podemos incluir información adicional (etiquetas) para conocer de qué tipo de modelo se trata.

![modelo](https://mlflow.org/docs/latest/assets/images/model-alias-and-tags-0318d486b2bf16992f488de5a00ce474.png)

Cualquier modelo registrado es accesible una vez tenemos el servidor de MLFlow en marcha. De este modo podemos rescatar distintas versiones del modelo de una forma centralizada.

In [24]:
import mlflow.sklearn
from sklearn.datasets import make_regression

model_name = "sk-learn-random-forest-reg-model"
model_version = "1"

# Load the model from the Model Registry
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.sklearn.load_model(model_uri)

# Generate a new dataset for prediction and predict
X_new, _ = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
y_pred_new = model.predict(X_new)

print(y_pred_new)

[ 16.36355607 -20.09258424   8.0136586    6.16919118  -1.81185423
   4.03116362 -24.95801449  68.78053495 -45.0766513   64.44760141
 -40.16931792 -25.54191065 -14.39985794 -38.0567874    8.05358765
 -25.73029816 -15.91990041 -10.99985266 -24.2475118  -32.70582446
  17.34781751  68.49980732  44.5541425   41.31593646  48.16602726
 -23.62019943  47.15590018  69.12741949  48.16602726  -0.26024544
 -28.49126919 -10.99985266  10.73067585 -10.61092056  -4.7324722
   2.76556278  58.93099448 -31.19567455 -35.55773052 -23.99366895
  48.16602726  13.34984948  12.56552213 -18.66808469 -32.70582446
 -39.30386685 -34.29680647  48.44675489 -33.40149961  20.35083862
 -15.0214084  -34.55064932  -2.28963784 -19.61227378   7.6979477
 -25.86538741 -11.95702358 -15.36598686   5.88539811 -30.23881739
 -25.47645531 -43.61170248 -43.7442754  -14.59055495 -40.16931792
 -32.70582446  -2.68114572  -5.39418041  16.15991316  -2.28963784
  41.662821    10.04512765  51.22797543 -23.09874036  10.04512765
  46.5774364

## Ejemplo completo

Nuestro data scientist procede a obtener los datos y realizar su magia encontrando un modelo que devuelve buenos resultados.

In [25]:
import pandas as pd
from mlflow.models import infer_signature

# Load dataset
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

# Split the data into training, validation, and test sets
train, test = train_test_split(data, test_size=0.25, random_state=42)
train_x = train.drop(["quality"], axis=1).values
train_y = train[["quality"]].values.ravel()
test_x = test.drop(["quality"], axis=1).values
test_y = test[["quality"]].values.ravel()
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)
signature = infer_signature(train_x, train_y)

[Hyperopt](https://hyperopt.github.io/hyperopt/) es una alternativa a otros sistemas de búsqueda de hiperparámetros. Nos permite buscar una serie de hiperparámetros para nuestro modelo de forma eficiente y distribuida. Esto se vuelve muy importante cuando requerimos entrenar modelo pesado como las redes neuronales a escala.

In [26]:
%pip install hyperopt
%pip install -U git+https://github.com/hyperopt/hyperopt

Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/hyperopt/hyperopt to C:\Users\tifaw\AppData\Local\Temp\pip-req-build-lmz5nifg
  Resolved https://github.com/hyperopt/hyperopt to commit c49ad148201c81c6ad1b43730ef4d0912734aa37
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/hyperopt/hyperopt 'C:\Users\tifaw\AppData\Local\Temp\pip-req-build-lmz5nifg'


In [27]:
import keras
import numpy as np
from hyperopt import STATUS_OK

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):
    # Define model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)
    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    # Compile model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"], momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )

    # Train model with MLflow tracking
    with mlflow.start_run(nested=True):
        model.fit(
            train_x,
            train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
        )
        # Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64)
        eval_rmse = eval_result[1]

        # Log parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        # Log model
        mlflow.tensorflow.log_model(model, "model", signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

La función objetivo, como en todo proceso de optimización, guía cómo de bien estamos cambiando los parámetros de nuestro proceso. En este caso serán los hiperparámetros de nuestro entrenamiento (learning-rate y momentum).

In [28]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [29]:
from hyperopt import Trials, fmin, hp, tpe

space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0),
}

mlflow.set_experiment("wine-quality")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1779792820307, experiment_id='2', last_update_time=1779792820307, lifecycle_stage='active', name='wine-quality', tags={}, trace_location=None, workspace='default'>

In [30]:
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 21s 478ms/step - loss: 37.1970 - root_mean_squared_error: 6.0989
33/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 35.4266 - root_mean_squared_error: 5.9514   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 32.3320 - root_mean_squared_error: 5.6861 - val_loss: 28.1340 - val_root_mean_squared_error: 5.3041

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 29.0968 - root_mean_squared_error: 5.3941
22/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 27.1840 - root_mean_squared_error: 5.2132 
44/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 26.3019 - root_mean_squared_error: 5.1274
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 24.4927 - root_mean_squared_error: 4.9490 - val_loss: 21.3687 - val_root_mean_squared_error: 4.6226

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - loss: 21.0234 - root_mean_squared_error: 4.5

2026/05/26 12:58:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run stately-worm-846 at: http://localhost:5000/#/experiments/2/runs/507676d4c25f48d3ba211ecc9c3ce57d

🧪 View experiment at: http://localhost:5000/#/experiments/2

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 21s 480ms/step - loss: 38.7214 - root_mean_squared_error: 6.2226
35/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 14.7310 - root_mean_squared_error: 3.6857   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5.2334 - root_mean_squared_error: 2.2877 - val_loss: 1.2765 - val_root_mean_squared_error: 1.1298

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - loss: 1.7385 - root_mean_squared_error: 1.3185
23/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.2007 - root_mean_squared_error: 1.0932 
41/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1411 - root_mean_squared_error: 1.0663
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.0256 - root_mean_squared_

2026/05/26 12:59:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run shivering-shrike-589 at: http://localhost:5000/#/experiments/2/runs/5fc2597f3c5642fc9837f416f6646859

🧪 View experiment at: http://localhost:5000/#/experiments/2                  

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 21s 470ms/step - loss: 38.9166 - root_mean_squared_error: 6.2383
35/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 8.9768 - root_mean_squared_error: 2.8058    
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 2.9928 - root_mean_squared_error: 1.7300 - val_loss: 0.9726 - val_root_mean_squared_error: 0.9862

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 0.7314 - root_mean_squared_error: 0.8552
18/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7852 - root_mean_squared_error: 0.8858 
31/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.7844 - root_mean_squared_error: 0.8854
42/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.7

...

🏃 View run thoughtful-whale-445 at: http://localhost:5000/#/experiments/2/runs/5179c6aa16d143969d943f4c68bb2b05

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 29s 657ms/step - loss: 38.2597 - root_mean_squared_error: 6.1854
25/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38.6180 - root_mean_squared_error: 6.2143   
44/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38.6241 - root_mean_squared_error: 6.2148
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 38.6569 - root_mean_squared_error: 6.2175 - val_loss: 38.5228 - val_root_mean_squared_error: 6.2067

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - loss: 36.7010 - root_mean_squared_error: 6.0581
18/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 37.9587 - root_mean_squared_error: 6.1610 
33/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - lo

2026/05/26 13:00:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run zealous-bass-485 at: http://localhost:5000/#/experiments/2/runs/4f2037fbc39542158aa5bcf70d879d5c

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

100%|██████████| 8/8 [02:24<00:00, 18.03s/trial, best loss: 0.7613739967346191]

2026/05/26 13:01:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.029999951486323537), 'momentum': np.float64(0.07708035271564506)}
Best eval rmse: 0.7613739967346191
🏃 View run sassy-pig-911 at: http://localhost:5000/#/experiments/2/runs/b1dcde7f059f41d4b38aca65faeb8436
🧪 View experiment at: http://localhost:5000/#/experiments/2


Nuestro mejor RMSE es de 0.71 con los parámetros:

* learning-rate: 0.045
* momentum: 0.73

**NOTA**: Vuestro parámetros pueden variar ligeramente.

Verificad en el interfaz de MLFlow si esto es así. Podéis volver a ejecutar la celda y evaluar esta nueva ejecución.

In [31]:
mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 29s 649ms/step - loss: 40.2096 - root_mean_squared_error: 6.3411
14/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 36.3891 - root_mean_squared_error: 6.0313   
28/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 34.9918 - root_mean_squared_error: 5.9135
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 33.4764 - root_mean_squared_error: 5.7823
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 30.0135 - root_mean_squared_error: 5.4785 - val_loss: 24.2416 - val_root_mean_squared_error: 4.9236

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - loss: 26.5660 - root_mean_squared_error: 5.1542
24/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 23.5490 - root_mean_squared_error: 4.8507 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 19.8378 - root_mean_squared_error: 4.4540 - val_loss: 15.9986 - val_root_mean_squared_error: 3.9998

Epoch 3/3                                          

2026/05/26 13:01:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run fearless-ray-780 at: http://localhost:5000/#/experiments/2/runs/f9ac386d390e4692bcfb072f1cf91681

🧪 View experiment at: http://localhost:5000/#/experiments/2

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 23s 525ms/step - loss: 35.8799 - root_mean_squared_error: 5.9900
31/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8.8724 - root_mean_squared_error: 2.8059    
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 2.8504 - root_mean_squared_error: 1.6883 - val_loss: 1.0030 - val_root_mean_squared_error: 1.0015

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - loss: 1.1807 - root_mean_squared_error: 1.0866
20/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9465 - root_mean_squared_error: 0.9713 
42/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8878 - root_mean_squared_error: 0.9410
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.8028 - root_mean_squared_

2026/05/26 13:01:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run skillful-squid-254 at: http://localhost:5000/#/experiments/2/runs/4d576cd444f348b0893c7b796b1bc8de

🧪 View experiment at: http://localhost:5000/#/experiments/2                  

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 20s 458ms/step - loss: 41.5003 - root_mean_squared_error: 6.4421
35/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38.4500 - root_mean_squared_error: 6.1992   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 33.7187 - root_mean_squared_error: 5.8068 - val_loss: 26.9884 - val_root_mean_squared_error: 5.1950

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 28.7141 - root_mean_squared_error: 5.3586
38/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 24.2602 - root_mean_squared_error: 4.9233 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 21.4095 - root_mean_squared_error: 4.6270 - val_loss: 17.2423 - val_root_mean_squared_err

...

🏃 View run defiant-perch-513 at: http://localhost:5000/#/experiments/2/runs/e7866b9d322b496bae74a54621b9c4fb

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 29s 657ms/step - loss: 39.0702 - root_mean_squared_error: 6.2506
21/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 36.4080 - root_mean_squared_error: 6.0332   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 33.2959 - root_mean_squared_error: 5.7703 - val_loss: 30.2616 - val_root_mean_squared_error: 5.5011

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - loss: 28.9690 - root_mean_squared_error: 5.3823
17/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 29.5364 - root_mean_squared_error: 5.4347 
34/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 29.0782 - root_mean_squared_error: 5.3922
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss:

2026/05/26 13:03:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run shivering-ox-451 at: http://localhost:5000/#/experiments/2/runs/79bd81a53433436990e186cb63c07637

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

100%|██████████| 8/8 [02:34<00:00, 19.35s/trial, best loss: 0.7690587639808655]

2026/05/26 13:04:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.032558870786010705), 'momentum': np.float64(0.04916344868633815)}
Best eval rmse: 0.7690587639808655
🏃 View run lyrical-bird-979 at: http://localhost:5000/#/experiments/2/runs/3f44f8c44c9f4d20a3b31d18d3f76611
🧪 View experiment at: http://localhost:5000/#/experiments/2


Si estamos contentos con un modelo en concreto podemos proceder a registrarlo:

![registry](img/mlflowreg.png)

## Exponer modelo

MLFlow serving: https://mlflow.org/docs/latest/ml/deployment/

![serving](https://mlflow.org/docs/latest/assets/images/mlflow-deployment-overview-99db410b2c58fedf506eb9ce5aa41a86.png)

Una vez hecho esto es sencillo invocar al proceso que sirve el modelo desde la terminal. Para ello es necesario establecer la URL del servidor de tracking en una variable local previamente:

```
export MLFLOW_TRACKING_URI=http://localhost:5000
```

Puede que para la gestión del entorno os pida también incluir las librerías [pyenv](https://github.com/pyenv/pyenv) y virtualenv (`!pip install virtualenv`).

Una vez configurada vuestra máquina, se vuelve un proceso sencillo en el que poder invocar el comando siguiente para servir el modelo:

```
mlflow models serve -m "models:/<nombre del modelo>/1" --port 5002
```

In [32]:
import requests

url_modelo = "http://localhost:5002/invocations"

json_data = {"dataframe_split": {
                "columns": [
                    "fixed acidity","volatile acidity","citric acid","residual sugar","chlorides","free sulfur dioxide","total sulfur dioxide","density","pH","sulphates","alcohol"],
                    "data": [[7,0.27,0.36,20.7,0.045,45,170,1.001,3,0.45,8.8]]}
}
headers = {'Content-Type' : 'application/json'}

response = requests.post(url=url_modelo, headers=headers, json=json_data)
print(response.status_code)

ConnectionError: HTTPConnectionPool(host='localhost', port=5002): Max retries exceeded with url: /invocations (Caused by NewConnectionError("HTTPConnection(host='localhost', port=5002): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))

In [ ]:
response.content